[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [5]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [6]:
import torch
import torch.nn as nn
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # W_q, W_k, W_v, W_o
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads


    def forward(self, x_q, x_kv):
        # pass  # Q from x_q, K/V from x_kv, no causal mask
        #xq (B, S_q, D)
        # x_kv = (B, S_kv, D)
        B, S_q, D = x_q.shape
        _, S_kv, _ = x_kv.shape
        Q = self.W_q(x_q).view(B, S_q, self.num_heads, self.d_k).transpose(1, 2)
        #Q (B, num_heads, S_q, d_k)

        K = self.W_k(x_kv).view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x_kv).view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        #K, V (B, num_heads, S_kv, d_k)

        score = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        #score (B, num_heads, S_q, S_kv)

        att_w = torch.softmax(score, dim = -1)

        output = torch.matmul(att_w, V)
        # (B, num_heads, S_q, d_k) -> (B, S_q, d_model)
        output = output.transpose(1, 2).reshape(B, S_q, self.d_model)
        return self.W_o(output)


In [12]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.7ms)
  ✅ [2/4] Q and KV different lengths (0.9ms)
  ✅ [3/4] No causal mask — all KV affects all Q (37.8ms)
  ✅ [4/4] Gradient flow (20.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (61.1ms total)
  Progress saved. Run status() to see your dashboard.

